In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch 
import numpy as np
import random
torch.autograd.set_detect_anomaly(True)
torch.multiprocessing.set_sharing_strategy("file_descriptor")
seed = 140421
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Register Calib

In [ ]:
from detectron2.data.datasets.pano360 import CalibDataset

debug = True
calib_train = CalibDataset(
    train=True,
    json_name="datasets/pano360_crops_dataset_cvpr_myDistWider_train.json",
    logger=None,
    debug=debug,
)
calib_val = CalibDataset(
    train=False,
    json_name="datasets/pano360_crops_dataset_cvpr_myDistWider_train.json",
    logger=None,
    debug=debug,
)

In [ ]:
from detectron2.data import DatasetCatalog
pano_train_name = "Pano360_train"
pano_val_name = "Pano360_val"
DatasetCatalog.register(pano_train_name, calib_train)
DatasetCatalog.register(pano_val_name, calib_val)

# Register COCOScale

In [ ]:
from pathlib import Path

base_path = Path.cwd()
coco_path = base_path / "data" / "coco" 
coco_annotations_path = coco_path / "annotations" 
coco_keypoints_path = coco_annotations_path / "person_keypoints_train2017.json"
coco_scalenet_results_path = coco_path / "coco_results" 
coco_images_root_path =  coco_path / "train2017"

In [ ]:
from detectron2.data.datasets.coco_scale import COCOScale2017
coco_scale_train = COCOScale2017(
    debug=debug,
    camera_parameters_file_path=coco_scalenet_results_path / "yannick_results_train2017_filtered",
    coco_json_file_path=coco_keypoints_path,
    coco_image_root_path=coco_images_root_path,
    coco_scale_pickle_path=coco_scalenet_results_path / "results_with_kps_20200208_morethan2_2-8" / "pickle"
)

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets.builtin_meta import _get_builtin_metadata
coco_meta = _get_builtin_metadata("coco_person")
coco_scale_dataset_name = "COCOScale2017_train"
DatasetCatalog.register(coco_scale_dataset_name, coco_scale_train)
coco_scale_meta = MetadataCatalog.get(coco_scale_dataset_name).set(
    json_file=coco_keypoints_path, image_root=coco_images_root_path, evaluator_type="coco", **coco_meta,
    thing_dataset_id_to_contiguous_id = {1: 0}  # COCO ID 1 → internal ID 0
)

In [ ]:
pano_meta = MetadataCatalog.get(pano_train_name).set(
    json_file=coco_keypoints_path, image_root=coco_images_root_path, evaluator_type="coco", **coco_meta,
    thing_dataset_id_to_contiguous_id = {1: 0}  # COCO ID 1 → internal ID 0
)

# Use Both

In [ ]:
from detectron2.data.datasets.coco_scale import COCOScale2017Calib
coco_scale_calib_dataset = COCOScale2017Calib(calib_train, coco_scale_train) 
coco_scale_calib_dataset_name = "COCOScale2017Calib_train"
if coco_scale_calib_dataset_name in DatasetCatalog:
    DatasetCatalog.remove(coco_scale_calib_dataset_name)
DatasetCatalog.register(coco_scale_calib_dataset_name, coco_scale_calib_dataset)
meta = MetadataCatalog.get(coco_scale_calib_dataset_name).set(
    json_file=coco_keypoints_path, image_root=coco_images_root_path, evaluator_type="coco", **coco_meta,
    thing_dataset_id_to_contiguous_id = {1: 0}  # COCO ID 1 → internal ID 0
)

# Train

In [ ]:
import os
from detectron2.engine import HybridScaleTrainer
from detectron2 import model_zoo
from detectron2.config import get_cfg

cfg = get_cfg()
config_path = "COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml"
cfg.merge_from_file(model_zoo.get_config_file(config_path))
experiment_name = "test-debug-calibkpsbbox-dataset"
cfg.OUTPUT_DIR = os.path.join("output", experiment_name)
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_path)  # Let training initialize from model zoo
# cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.SOLVER.IMS_PER_BATCH = 1  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.00025  # pick a good LR
cfg.MODEL.KEYPOINT_ON = True
cfg.MODEL.HEIGHT_ON = True
cfg.MODEL.HEIGHT_REFINE_ON = False
cfg.MODEL.META_ARCHITECTURE = "GeneralizedCamRCNN"
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # Number of classes
cfg.MODEL.ROI_KEYPOINT_HEAD.NUM_KEYPOINTS = 17 # Number of keypoints
cfg.MODEL.ROI_HEADS.NAME = "HeightStandardROIHeads"
cfg.MODEL.ROI_KEYPOINT_HEAD.NAME = "KRCNNConvDeconvUpsampleHeadHeightPred"
cfg.VIS_PERIOD = 10
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS=False
cfg.SOLVER.MAX_ITER = 500
cfg.SOLVER.AMP.ENABLED = True  # Enable AMP here -- improve 10s per iter approx.
# cfg.DATALOADER
cfg.DATASETS.TRAIN = (coco_scale_calib_dataset_name, )
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 4
cfg.DATALOADER.ASPECT_RATIO_GROUPING = False  # Doesn't work at the current implementation of MixedDataset

# SVMIW Losses
cfg.MODEL.ROI_KEYPOINT_HEAD.LOSS_WEIGHT = 10  # alpha_4
cfg.MODEL.ROI_BOX_HEAD.BBOX_REG_LOSS_WEIGHT = 10  # alpha_5


In [ ]:
trainer = HybridScaleTrainer(cfg) 
trainer.resume_or_load(resume=False)
trainer.train()